In [36]:
# Needed Packages
# pip install transformers sentencepiece torch os subprocess pathlib

In [37]:
import os
import subprocess
from pathlib import Path

def find_repo_root(repo_name="CS329"):
    try:
        root = subprocess.check_output(
            ["git", "rev-parse", "--show-toplevel"],
            stderr=subprocess.DEVNULL,
            text=True
        ).strip()
        root = Path(root)
        if root.name == repo_name:
            return root
    except Exception:
        pass

    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if p.name == repo_name and (p / ".git").exists():
            return p
        candidate = p / repo_name
        if candidate.exists() and (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(f"Could not find repo '{repo_name}'")

repo_root = find_repo_root("CS329")
os.chdir(repo_root)
print("Using repo:", repo_root)

Using repo: C:\Users\Abasu\Documents\GitHub\CS329


In [38]:
from Dialect_Detector import detect_dialect

In [39]:
# Load the OPUS-MT Arabic→English Model

from transformers import MarianMTModel, MarianTokenizer
import torch

MODEL_NAME = "Helsinki-NLP/opus-mt-ar-en"

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)




Loading weights: 100%|██████████| 258/258 [00:00<00:00, 37222.43it/s]


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(62834, 512, padding_idx=62833)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(62834, 512, padding_idx=62833)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [40]:
# Dialect Normalization Layer (Key Innovation)
# Simple normalization dictionaries per dialect
DIALECT_NORMALIZATION = {
    "egyptian": {
        "رايح": "ذاهب",
        "عايز": "أريد",
        "مش": "ليس",
        "فين": "أين",
        "ليه": "لماذا"
    },
    "levantine": {
        "راح": "ذهب",
        "بدّي": "أريد",
        "مو": "ليس",
        "وين": "أين"
    },
    "gulf": {
        "وين": "أين",
        "أبغى": "أريد",
        "مو": "ليس"
    }
}

def normalize_dialect(text: str, dialect: str):
    """
    Replace dialectal words with MSA equivalents.
    Returns normalized text + list of applied rules.
    """
    applied_rules = []

    rules = DIALECT_NORMALIZATION.get(dialect, {})
    tokens = text.split()

    normalized_tokens = []
    for token in tokens:
        if token in rules:
            normalized_tokens.append(rules[token])
            applied_rules.append(f"{token} → {rules[token]}")
        else:
            normalized_tokens.append(token)

    normalized_text = " ".join(normalized_tokens)
    return normalized_text, applied_rules


In [41]:
# Translation Function (OPUS-MT Inference)

def translate_ar_to_en(text: str, max_length: int = 128):
    """
    Translate Arabic text to English using OPUS-MT.
    """
    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        translated = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            early_stopping=True
        )

    return tokenizer.decode(translated[0], skip_special_tokens=True)


In [42]:
# Explanation Layer (Ambiguity Awareness)
# This is post-processing, not modeling

AMBIGUOUS_WORDS = {
    "راح": ["went", "will go"],
    "كان": ["was", "used to be"],
    "طلع": ["went out", "turned out", "appeared"]
}

def get_ambiguity_notes(text: str):
    notes = []
    for word, meanings in AMBIGUOUS_WORDS.items():
        if word in text:
            notes.append({
                "word": word,
                "possible_meanings": meanings
            })
    return notes


In [43]:
# Main Pipeline (This Is What You Demo)
# Assumes You already have detect_dialect(text) implemented.


def dialect_aware_translate(text: str, detect_dialect):
    """
    Full pipeline:
    dialect detection → normalization → translation → explanations
    """
    dialect = detect_dialect(text)

    normalized_text, applied_rules = normalize_dialect(text, dialect)

    translation = translate_ar_to_en(normalized_text)

    ambiguity_notes = get_ambiguity_notes(text)

    return {
        "input_text": text,
        "detected_dialect": dialect,
        "normalized_text": normalized_text,
        "applied_normalization_rules": applied_rules,
        "translation": translation,
        "ambiguities": ambiguity_notes
    }


In [44]:
# Example Run (Perfect for Live Demo)

example = "أنا رايح البيت ومش تعبان"

result = dialect_aware_translate(example, detect_dialect_dummy)

for k, v in result.items():
    print(f"{k}: {v}")


NameError: name 'detect_dialect_dummy' is not defined